In [1]:
import numpy as np

width_block_count = 6
height_block_count = 4

def slice_matrix(shape, threshold=0.5):
    random_matrix = np.random.normal(-0.3, 1, size=shape)
    rounded_matrix = (random_matrix>threshold).astype(int)
    return rounded_matrix

def revised_slice_matrix(shape, threshold=0.5):
    while True:
        random_matrix = np.random.normal(-0.3, 1, size=shape)
        rounded_matrix = (random_matrix>threshold).astype(int)

        col_sums = np.sum(rounded_matrix, axis=0)
        row_sums = np.sum(rounded_matrix, axis=1)
        if not np.any(row_sums==shape[1]) and not np.any(col_sums==shape[0]) and np.any(rounded_matrix):
            return rounded_matrix

def find_indices_of_ones(matrix):
    indices = np.where(matrix == 1)
    return list(zip(indices[0], indices[1]))


s_matrix = revised_slice_matrix((height_block_count, width_block_count))
idxs = find_indices_of_ones(s_matrix)
print(s_matrix)
print(idxs)

[[0 0 0 1 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [1 0 0 1 0 0]]
[(0, 3), (3, 0), (3, 3)]


In [5]:
import numpy as np
import cv2
import glob
from PIL import Image
from scipy import spatial

def montage(input_pic, tile_photos, output_file, tile_size):

    #Get all tiles
    tile_paths = []
    for file in glob.glob(tile_photos):
        tile_paths.append(file)

    # Import and resize all tiles
    tiles = []
    for path in tile_paths:
        tile = Image.open(path)
        tile = tile.resize(tile_size)
        tile = tile.convert("RGB")  #要不然有一些的image mode變成P，結果沒辦法用rgb下去算
        tiles.append(tile)

    # mode ref: https://pillow.readthedocs.io/en/stable/handbook/concepts.html#concept-modes

    # Calculate dominant color
    colors = []
    for tile in tiles:
        mean_color = np.array(tile).mean(axis=0).mean(axis=0)
        colors.append(mean_color)

    # Pixelate (resize) main photo
    main_photo = Image.open(input_pic)

    width = int(np.round(main_photo.size[0] / tile_size[0]))
    height = int(np.round(main_photo.size[1] / tile_size[1]))

    resized_photo = main_photo.resize((width, height))

    # Find closest tile photo for every pixel
    # Create a KDTree
    tree = spatial.KDTree(colors) ## 會報錯 --> 發現是colors裡面有些會變"一個數字"，才知道是原圖有些的 image mode 是rgb, rgba, p --> 是 p 會變成一個數字

    # Empty integer array to store indices of tiles
    closest_tiles = np.zeros((width, height), dtype=np.uint32)

    for i in range(width):
        for j in range(height):
            pixel = resized_photo.getpixel((i, j))  # Getthe pixel color at (i, j)
            closest = tree.query(pixel)             # Returns (distance, index)
            closest_tiles[i, j] = closest[1]        # We only need the index


    # Create an output image
    output = Image.new('RGB', main_photo.size)

    # Draw tiles
    for i in range(width):
        for j in range(height):
            # Offset of tile
            x, y = i*tile_size[0], j*tile_size[1]
            # Index of tile
            index = closest_tiles[i, j]
            # Draw tile
            output.paste(tiles[index], (x, y))

    output.save(output_file)



In [6]:
import cv2


def crop_img(original_img_path, result_img_path):
    img = cv2.imread(original_img_path, -1)
    h, w = img.shape[:2] 

    h_part = round((h-1)/height_block_count)
    w_part = round((w-1)/width_block_count)

    result = np.ones(((h_part)*height_block_count-2, (w_part)*width_block_count-2, 3), dtype=np.uint8)*255

    for i in range(height_block_count):
        for j in range(width_block_count):
            y_start = i*h_part
            y_end = (i+1)*h_part
            x_start = j*w_part
            x_end = (j+1)*w_part

            if (i, j) in idxs:
                continue
            else:
                tag = img[y_start:y_end-2, x_start:x_end-2]
                result[y_start:y_end-2, x_start:x_end-2 ] = tag

    # cv2.imshow('Image', result)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    cv2.imwrite(result_img_path, result)

# Can't do it
# crop_img('assets/hw2_pic1.jpg', 'results/hw3_output.jpg')
# montage('results/hw3_output.jpg',"mosaic-master\\mosaic-master\\dataset\\*","results/hw3_output_mont.jpg", (10, 10))    

c:\Users\soarb\.conda\envs\jupyter_server\lib\site-packages\PIL\Image.py:970: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [ ]:
montage('assets/hw2_pic1.jpg',"mosaic-master\\mosaic-master\\dataset\\*","results/hw3_mont2_output.jpg", (10, 10))    
crop_img('results/hw3_mont2_output.jpg', 'results/hw3_output_revised.jpg')
